# לאה — תיקון גוף (חזה לפי קנון) — קולאב
1. למעלה: Runtime → Change runtime type → T4 GPU
2. אחר כך: Runtime → Run all. בסוף יודפסו לינקים — העתק אותם לשימי פיתוח. הקבצים גם בסרגל Files משמאל.


In [ ]:
import os, subprocess, json, urllib.request
subprocess.run(["pip","install","-q","diffusers","transformers","accelerate","safetensors","opencv-python-headless"], timeout=600)
import torch, numpy as np, cv2
from PIL import Image
from diffusers import StableDiffusionXLInpaintPipeline, EulerAncestralDiscreteScheduler
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEV=="cuda" else "אין! הפעל T4")

BASE = "https://aidoor.shimi.land/gallery/lea-synth/"
pipe = StableDiffusionXLInpaintPipeline.from_pretrained("SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEV)

P = ("enormous heavy natural round breasts, each the size of a head, filling the whole chest width, "
     "full wide bottom, natural drop, pressed together with deep narrow cleavage, realistic natural areolas and nipples, "
     "smooth continuous skin, photorealistic")
NEG = ("small breasts, tiny breasts, flat chest, bra, lace, clothing, crease, fold, cut, deformed, "
       "lowres, bad anatomy, blurry, watermark")

def mask_for(w,h):
    m = np.zeros((h,w), np.uint8)
    cv2.ellipse(m,(int(w*0.54),int(h*0.50)),(int(w*0.22),int(h*0.085)),0,0,360,255,-1)
    return cv2.GaussianBlur(m,(0,0),max(3,w//90))

os.makedirs("/content/out", exist_ok=True)
for fname, seed in [("lea_synth_160001.png",220001),("lea_synth_160002.png",220002),
                    ("lea_synth_160003.png",220003),("lea_synth_160004.png",220004)]:
    urllib.request.urlretrieve(BASE+fname, "/content/"+fname)
    img = Image.open("/content/"+fname).convert("RGB")
    W,Hh = (img.size[0]//8)*8, (img.size[1]//8)*8
    img = img.resize((W,Hh), Image.LANCZOS)
    m = Image.fromarray(mask_for(W,Hh))
    g = torch.Generator(DEV).manual_seed(seed)
    res = pipe(prompt=P, negative_prompt=NEG, image=img, mask_image=m, strength=0.68,
               num_inference_steps=36, guidance_scale=6.0, generator=g).images[0]
    fp = f"/content/out/lea_chestfix_{seed}.png"
    res.save(fp); print("נשמר:", fp, flush=True)

import requests
for f in sorted(os.listdir("/content/out")):
    fp = "/content/out/"+f
    url = None
    try:
        r = requests.post("https://catbox.moe/user/api.php", data={"reqtype":"fileupload"}, files={"fileToUpload":open(fp,"rb")}, timeout=300)
        if r.ok and r.text.startswith("http"): url = r.text.strip()
    except Exception as e: print("catbox נכשל:", e)
    if not url:
        try:
            r = requests.post("https://tmpfiles.org/api/v1/upload", files={"file":open(fp,"rb")}, timeout=300)
            if r.ok: url = "https://tmpfiles.org/dl/"+r.json()["data"]["url"].split("/")[-1]
        except Exception as e: print("tmpfiles נכשל:", e)
    print(f, "→", url or "העלה ידנית מהסרגל Files", flush=True)
print("סיימנו. העתק לשימי את הלינקים שיצאו.", flush=True)